In [ ]:
import pandas as pd
import numpy as np
import math
import os
import glob
import uproot as r
import awkward as ak
import vector
vector.register_awkward()
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# 自定义参数
custom_signal_processes = ['a10', 'a30', 'a50', 'a70', 'a85']
custom_background_processes = ['atata']
custom_scale_factor = 1000
custom_labels_list = [r'$m_{a} = 10$ GeV', r'$m_{a} = 30$ GeV', r'$m_{a} = 50$ GeV', r'$m_{a} = 70$ GeV', r'$m_{a} = 85$ GeV', 'BG']
custom_cs_list = [8.587, 0.98, 0.18, 0.019, 4.1*10**(-4), 84.29]
custom_colors = ['black', 'blue', 'red', 'green', 'magenta', 'cyan']

In [ ]:
a_10 = pd.read_csv('./train_data/a10.csv')
a_30 = pd.read_csv('./train_data/a30.csv')
a_50 = pd.read_csv('./train_data/a50.csv')
a_70 = pd.read_csv('./train_data/a70.csv')
a_85 = pd.read_csv('./train_data/a85.csv')
atata = pd.read_csv('./train_data/atata.csv')

In [ ]:
data_dict = {
    R'$m_{a} = 10$ GeV': a_10,        # 信号类
    R'$m_{a} = 30$ GeV': a_30,        # 信号类
    R'$m_{a} = 50$ GeV': a_50,        # 信号类
    R'$m_{a} = 70$ GeV': a_70,        # 信号类
    R'$m_{a} = 85$ GeV': a_85,        # 信号类
    R'$\gamma\tau\tau$': atata,      # 背景类
}

In [ ]:
def sanitize_filename(filename):
    # 替换特殊字符为下划线
    return filename.replace('$', '').replace('{', '').replace('}', '').replace('\\', '').replace('/', '_')

def compute_invariant_mass_and_save(data_dict, particle_type='Electron'):
    """
    计算每个数据集的不变质量，并将结果添加到原始数据中，最后写入新的CSV文件。
    
    参数:
        data_dict (dict): 包含数据的字典，键为标签，值为 DataFrame。
        output_file_prefix (str): 输出CSV文件的前缀名称。
        particle_type (str): 粒子类型，默认为'Electron'。
    """
    # 定义质量
    Electron_mass = 0.000511
    Electron_mass = 0.105

    # 根据粒子类型选择质量
    if particle_type == 'Electron':
        particle_mass = Electron_mass
    elif particle_type == 'Electron':
        particle_mass = Electron_mass
    else:
        particle_mass = f'^{particle_type}_mass'

    # 遍历每个数据集并处理
    for key, df in data_dict.items():
        # 获取四维动量的分量
        pt = df.filter(regex=f'^{particle_type}_PT').values
        eta = df.filter(regex=f'^{particle_type}_Eta').values
        phi = df.filter(regex=f'^{particle_type}_Phi').values
        
        # 确保 mass 数组的长度与其它数据一致
        mass = np.full_like(pt, particle_mass, dtype=float)  # 填充 mass 数组，确保与 pt, eta, phi 长度一致
        #如果是jet拿着部分代码应该为mass = df.filter(regex=f'^{particle_type}_mass').values

        # 创建四维动量向量
        particles = ak.zip({
            "pt": pt,
            "eta": eta,
            "phi": phi,
            "mass": mass,
        }, with_name="Momentum4D")

        # 计算两个μ子的四维动量和不变质量
        lead_Electron = particles[:, 0]  # 第一个μ子
        nextlead_Electron = particles[:, 1]  # 第二个μ子
        
        # 使用 vector 库的加法操作符进行四维动量加法
        diElectron = lead_Electron + nextlead_Electron  # 计算两个μ子的四维动量和
        
        # 提取不变质量
        diElectron_mass = diElectron.mass  # 计算不变质量
        
        # 直接使用 vector 库提供的属性计算能量和动量分量
        lead_Electron_energy = lead_Electron.e
        lead_Electron_px = lead_Electron.px
        lead_Electron_py = lead_Electron.py
        lead_Electron_pz = lead_Electron.pz

        nextlead_Electron_energy = nextlead_Electron.e
        nextlead_Electron_px = nextlead_Electron.px
        nextlead_Electron_py = nextlead_Electron.py
        nextlead_Electron_pz = nextlead_Electron.pz

        # 获取光子的四维动量分量
        photon_pt = df.filter(regex=f'^Photon_PT').values
        photon_eta = df.filter(regex=f'^Photon_Eta').values
        photon_phi = df.filter(regex=f'^Photon_Phi').values
        photons = ak.zip({
            "pt": photon_pt,
            "eta": photon_eta,
            "phi": photon_phi,
            "mass": np.zeros_like(photon_pt),  # 光子的质量为零,
        }, with_name="Momentum4D")

        lead_photon = photons[:,0]

        # 获取光子的能量和动量分量
        photon_energy = photons.e
        photon_px = photons.px
        photon_py = photons.py
        photon_pz = photons.pz

        # 将计算结果添加到数据中
        df['lead_Electron_energy'] = lead_Electron_energy
        df['lead_Electron_px'] = lead_Electron_px
        df['lead_Electron_py'] = lead_Electron_py
        df['lead_Electron_pz'] = lead_Electron_pz
        df['nextlead_Electron_energy'] = nextlead_Electron_energy
        df['nextlead_Electron_px'] = nextlead_Electron_px
        df['nextlead_Electron_py'] = nextlead_Electron_py
        df['nextlead_Electron_pz'] = nextlead_Electron_pz
        df['diElectron_mass'] = diElectron_mass
        df['deltar_mu_0_mu_1'] = lead_Electron.deltaR(nextlead_Electron)
        df['deltar_mu_0_gamma'] = lead_Electron.deltaR(lead_photon)
        df['deltar_mu_1_gamma'] = nextlead_Electron.deltaR(lead_photon)
        
        # 添加光子的能量和动量分量
        df['photon_energy'] = photon_energy[:, 0]
        df['photon_px'] = photon_px[:, 0]
        df['photon_py'] = photon_py[:, 0]
        df['photon_pz'] = photon_pz[:, 0]

        # 确保目录存在
        output_dir = './train_data/select_data'
        os.makedirs(output_dir, exist_ok=True)

        # 使用 sanitize_filename() 处理文件名
        sanitized_key = sanitize_filename(key)  # 处理字典键，移除特殊字符
        output_file = os.path.join(output_dir, f'{sanitized_key}.csv')
        
        # 保存数据到新的CSV文件
        df.to_csv(output_file, index=False)
        print(f"Saved {output_file}")

In [ ]:
compute_invariant_mass_and_save(data_dict, particle_type='Electron')